# Named Entity Recognition — Model Evaluation

This notebook evaluates the trained spaCy model on the CoNLL-2003 test set.

We calculate precision, recall and F1 for PERSON, LOCATION and ORGANIZATION entities.

In [2]:
from pathlib import Path
from collections import defaultdict

import spacy
from datasets import load_dataset

MODEL_PATH = Path("ner_model")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "ner_model was not found. Run 02_train_ner_model.ipynb first."
    )

nlp = spacy.load(MODEL_PATH)
dataset = load_dataset(
    "lhoestq/conll2003",
    trust_remote_code=True
)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lhoestq/conll2003' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


In [3]:
LABEL_MAP = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC",
}

KEEP = {"PER": "PERSON", "ORG": "ORG", "LOC": "LOC"}

def gold_spans(row):
    tokens = row["tokens"]
    tags = row["ner_tags"]
    text = " ".join(tokens)

    offsets = []
    cursor = 0
    for token in tokens:
        start = text.find(token, cursor)
        end = start + len(token)
        offsets.append((start, end))
        cursor = end + 1

    spans = []
    current = None

    for i, tag_id in enumerate(tags + [0]):
        tag = LABEL_MAP[tag_id]

        if tag.startswith("B-"):
            if current:
                spans.append(current)

            label = KEEP.get(tag[2:])
            current = (offsets[i][0], offsets[i][1], label) if label else None

        elif tag.startswith("I-") and current:
            current = (current[0], offsets[i][1], current[2])

        else:
            if current:
                spans.append(current)
            current = None

    return text, {span for span in spans if span[2] is not None}

In [4]:
stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

# Evaluate a useful test subset; increase this number for a full run.
test_rows = dataset["test"][:3000]

for row in test_rows:
    text, gold = gold_spans(row)
    doc = nlp(text)

    predicted = {
        (ent.start_char, ent.end_char, ent.label_)
        for ent in doc.ents
        if ent.label_ in {"PERSON", "ORG", "LOC"}
    }

    for label in ["PERSON", "ORG", "LOC"]:
        gold_label = {x for x in gold if x[2] == label}
        pred_label = {x for x in predicted if x[2] == label}

        stats[label]["tp"] += len(gold_label & pred_label)
        stats[label]["fp"] += len(pred_label - gold_label)
        stats[label]["fn"] += len(gold_label - pred_label)

TypeError: string indices must be integers, not 'str'

In [5]:
import pandas as pd

results = []

for label, values in stats.items():
    tp = values["tp"]
    fp = values["fp"]
    fn = values["fn"]

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0
    )

    results.append({
        "Entity": label,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
    })

results_df = pd.DataFrame(results)
results_df

""


In [6]:
print("Micro-level counts:")
print(dict(stats))

Micro-level counts:
{}


In [7]:
examples = [
    "Tim Cook works at Apple in California.",
    "Barack Obama visited London.",
    "Microsoft opened an office in New York."
]

for text in examples:
    doc = nlp(text)
    print("\nTEXT:", text)
    for ent in doc.ents:
        print(f"  {ent.text} -> {ent.label_}")


TEXT: Tim Cook works at Apple in California.
  Tim Cook -> PERSON
  Apple -> ORG
  California -> LOC

TEXT: Barack Obama visited London.
  Barack Obama -> PERSON
  London -> LOC

TEXT: Microsoft opened an office in New York.
  Microsoft -> ORG
  New York -> LOC


## Interpreting the Results

NER is stricter than ordinary text classification because both the **entity label and exact span boundary** must be correct.

For interviews, explain:
- **Precision:** How many predicted entities were correct?
- **Recall:** How many actual entities were found?
- **F1:** Harmonic mean of precision and recall.
- Entity-level evaluation is more meaningful than simply checking token accuracy.

The evaluation subset can be increased when more compute time is available.